# EDA 

In [ ]:
import h5py
import numpy as np
import torch

# 1. Paths to the P-CAM dataset
# Update these paths to point to your local extracted data directory
train_x_path = 'data/camelyonpatch_level_2_split_train_x.h5'
train_y_path = 'data/camelyonpatch_level_2_split_train_y.h5'

# 2. Open the HDF5 files in 'read' mode
h5_train_x = h5py.File(train_x_path, 'r')
h5_train_y = h5py.File(train_y_path, 'r')

x_dataset = h5_train_x['x']
y_dataset = h5_train_y['y']

print(f"Full training data shape: {x_dataset.shape}")

# 3. Load the full dataset into memory
# Loading all 262,144 images into RAM requires significant memory.
# PyTorch DataLoader to stream batches directly from disk.
full_images = x_dataset[:]
full_labels = y_dataset[:].flatten() 

# 4. Convert to PyTorch tensors and normalize (0.0 to 1.0)
tensor_x = torch.tensor(full_images, dtype=torch.float32) / 255.0
tensor_y = torch.tensor(full_labels, dtype=torch.long)

# 5. Permute dimensions for PyTorch to (Batch, Channels, Height, Width)
tensor_x = tensor_x.permute(0, 3, 1, 2)

print(f"Final Tensor X shape: {tensor_x.shape}")
print(f"Final Tensor Y shape: {tensor_y.shape}")

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Geometric: horizontal/vertical flips and strict 90-degree rotations
train_transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    # Enforce strict 90-degree increments
    T.RandomChoice([
        T.RandomRotation((0, 0)),
        T.RandomRotation((90, 90)),
        T.RandomRotation((180, 180)),
        T.RandomRotation((270, 270))
    ]),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2) 
])

class PCamDataset(Dataset):
    def __init__(self, x_tensor, y_tensor, transform=None):
        self.x = x_tensor
        self.y = y_tensor
        self.transform = transform
        
    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, idx):
        img = self.x[idx]
        label = self.y[idx]
        
        # Apply the transformations if specified
        if self.transform:
            img = self.transform(img)
            
        return img, label

# Wrapping the full tensors and applying the exact transforms
full_dataset = PCamDataset(tensor_x, tensor_y, transform=train_transforms)

full_loader = DataLoader(full_dataset, batch_size=128, shuffle=True)

print(f"Total batches in full data loader: {len(full_loader)}")

# 4. Visual Check: Fetch a single batch and display one augmented image
images, labels = next(iter(full_loader))
single_img = images[0].permute(1, 2, 0).numpy() # Permute back to (H, W, C)

plt.figure(figsize=(4, 4))
plt.imshow(single_img)
plt.title(f"Augmented Image | Label: {labels[0].item()}")
plt.axis('off')
plt.show()